In [6]:
import pandas as pd
import s3fs

# Define connection details
minio_endpoint = 'http://localhost:9000'
access_key = 'admin'
secret_key = 'Admin123!'

# Create the filesystem object
fs = s3fs.S3FileSystem(
    key=access_key,
    secret=secret_key,
    client_kwargs={'endpoint_url': minio_endpoint}
)

In [7]:
files = fs.glob("s3://bronze/ecommerce_events/**/*.parquet")

bronze_df = pd.read_parquet(
    files,
    filesystem=fs,
    engine="pyarrow"
)

bronze_df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,kafka_ts,kafka_partition,kafka_offset,ingested_at,date_partition
0,2020-01-01 00:00:00 UTC,view,1005073,2232732093077520756,construction.tools.light,samsung,1130.02,519698804,69b5d72f-fd6e-4fed-aa23-1286b2ca89a0,2026-04-30 16:38:27.593,0,0,2026-05-04 11:08:10.166902,2026-04-30
1,2020-01-01 00:00:02 UTC,view,16500026,2232732108713886406,apparel.costume,gamma,37.35,581037554,c161400e-630b-4b59-8797-d9b9714444bf,2026-04-30 16:38:27.593,0,1,2026-05-04 11:08:10.166902,2026-04-30
2,2020-01-01 00:00:02 UTC,view,4802273,2232732079706079299,sport.bicycle,samsung,6.64,595414563,176fd102-7b61-4452-a0e5-f1f8cc9b4b95,2026-04-30 16:38:27.593,0,2,2026-05-04 11:08:10.166902,2026-04-30
3,2020-01-01 00:00:03 UTC,view,26000317,2232732082474320004,,,11.84,592648495,7294aca4-58be-4fbe-9a4c-7f2b528b3fad,2026-04-30 16:38:27.593,0,3,2026-05-04 11:08:10.166902,2026-04-30
4,2020-01-01 00:00:04 UTC,view,21407348,2232732082063278200,electronics.clocks,casio,122.01,578858757,bdf051a8-1594-4630-b93d-2ba62b92d039,2026-04-30 16:38:27.593,0,4,2026-05-04 11:08:10.166902,2026-04-30


In [15]:
bronze_df[~bronze_df['product_id'].str.isdigit()]

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,kafka_ts,kafka_partition,kafka_offset,ingested_at,date_partition


In [8]:
# 1. Đếm số lớp dựa trên dấu chấm
num_layers = bronze_df['category_code'].fillna('').str.count('.').add(1)

# 2. Tạo điều kiện lọc: Bỏ khoảng trắng, kiểm tra xem có bị rỗng hoặc null không
is_empty = bronze_df['category_code'].str.strip().isin(['', None]) | bronze_df['category_code'].isna()

# 3. Dòng nào thỏa mãn is_empty thì gán bằng 0
num_layers = num_layers.where(~is_empty, 0)

print(num_layers.value_counts().sort_index())


category_code
0     1888
10     388
11      88
12      32
13      60
14    3276
15     256
16     328
17      96
18     164
19    2276
20      12
21     140
22    2180
23     608
24     360
25    6852
26     644
27     372
28    1040
29     448
30     716
31     352
32     152
33     880
34     368
36      16
39       4
Name: count, dtype: int64


In [9]:
bronze_df['category_code'].isna().sum()

np.int64(0)

In [10]:
bronze_df['category_code'].isin(['', ' ']).sum()

np.int64(1888)

In [11]:
silver_files = fs.glob("s3://silver/ecommerce_events/**/*.parquet")

silver_df = pd.read_parquet(
    silver_files,
    filesystem=fs,
    engine="pyarrow"
)

In [12]:
silver_df['category_code'].isin(['', ' ']).sum()

np.int64(0)

In [13]:
silver_df.columns

Index(['source_event_id', 'event_ts', 'event_year', 'event_month', 'event_day',
       'event_hour', 'event_type', 'product_id', 'category_id',
       'category_code', 'category_l1', 'category_l2', 'category_l3', 'brand',
       'price', 'user_id', 'user_session', 'kafka_ts', 'kafka_partition',
       'kafka_offset', 'bronze_ingested_at', 'silver_processed_at', 'is_valid',
       'invalid_reason', 'event_date'],
      dtype='str')

In [14]:
silver_df.isna().sum()

source_event_id           0
event_ts                  0
event_year                0
event_month               0
event_day                 0
event_hour                0
event_type                0
product_id                0
category_id               0
category_code           472
category_l1             472
category_l2             472
category_l3            2241
brand                     0
price                     0
user_id                   0
user_session              0
kafka_ts                  0
kafka_partition           0
kafka_offset              0
bronze_ingested_at        0
silver_processed_at       0
is_valid                  0
invalid_reason         5999
event_date                0
dtype: int64